<div style="background: linear-gradient(135deg, #fdfbfb 0%, #ebedee 100%); border-radius: 10px; border-left: 6px solid #ff9900; box-shadow: 0 3px 6px rgba(0,0,0,0.06); width: 100%; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; overflow: hidden;">
  <table style="border: none; width: 100%; margin: 0; border-collapse: collapse; background: transparent;">
    <tr style="border: none; background: transparent;">
      <td style="border: none; width: 100px; text-align: center; vertical-align: middle; padding: 14px 0 14px 15px; background: transparent;">
        <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
      </td>
      <td style="border: none; vertical-align: middle; padding: 14px 15px; background: transparent;">
        <div style="color: #555555; font-size: 13px; font-weight: 500; margin-bottom: 2px;">Excerpt from the Amazon book</div>
        <div style="font-family: 'Arial Black', Impact, sans-serif; font-size: 18px; font-weight: 900; text-transform: uppercase; letter-spacing: 1.5px; color: #111111; margin: 6px 0 10px 0; line-height: 1.2;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</div>
        <div>
          <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: none; font-weight: 600; font-size: 13px; display: inline-block;">🛒 Buy on Amazon &rarr;</a>
        </div>
      </td>
    </tr>
  </table>
</div>

In [1]:
# 1. getting files from repository
!git clone https://github.com/sansalamanca/shining-path_DP700.git


Cloning into 'shining-path_DP700'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 71 (delta 28), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 565.39 KiB | 4.49 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
# initiate pyspark session
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("sessión_Colab_9") \
    .getOrCreate()


In [3]:
#  Reading a parquet file will include its shemaaspark.read.format("parquet").load(
df = spark.read.format("parquet").load(
    "shining-path_DP700/data/orders_parquet"
)

df.show()

+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|         

# **Split column CustomerName in two columns (FirstName and LastName)**

In [4]:
from pyspark.sql.functions import *
# Create Year and Month columns
transformed_df = df.withColumn("Year", year(col("OrderDate"))).withColumn("Month", month(col("OrderDate")))
# Create the new FirstName and LastName fields
transformed_df = transformed_df.withColumn("FirstName", split(col("CustomerName"), " ").getItem(0)).withColumn("LastName", split(col("CustomerName"), " ").getItem(1))
# Filter and reorder columns
transformed_df = transformed_df["SalesOrderNumber", "SalesOrderLineNumber", "OrderDate", "Year", "Month", "FirstName", "LastName", "Email", "Item", "Quantity", "UnitPrice", "Tax"]
# Display orders
transformed_df.show()   # En Fabric use:  display(transformed_df.limit(5))


+----------------+--------------------+----------+----+-----+---------+---------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|Year|Month|FirstName| LastName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+----+-----+---------+---------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|2019|    7|  Christy|      Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|2019|    7|    Julio|     Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|2019|    7|   Curtis|       Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|2019|    7|    Ruben|   Prasad|ruben10